In [2]:
import pathlib
import re
import sys
import pandas as pd

_p = pathlib.Path.cwd()
while _p != _p.parent and not (_p / "db" / "dataraflow_db.py").exists():
  _p = _p.parent
sys.path.insert(0, str(_p))
from db.dataraflow_db import get_connection, get_engine, run_query

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

_BLOCKED = re.compile(
    r"\b(INSERT|UPDATE|DELETE|MERGE|DROP|ALTER|CREATE|TRUNCATE|EXEC|EXECUTE|GRANT|REVOKE|DENY|INTO|BACKUP|RESTORE)\b",
    re.IGNORECASE,
)


def q(sql: str) -> pd.DataFrame:
  """Run a read-only query and return a DataFrame."""
  scrubbed = re.sub(r"'([^']|'')*'", "''", sql)
  scrubbed = re.sub(r"--[^\n]*", "", scrubbed)
  scrubbed = re.sub(r"/\*.*?\*/", "", scrubbed, flags=re.S)
  hit = _BLOCKED.search(scrubbed)
  if hit:
    raise PermissionError(
        f"'{hit.group(0).upper()}' is not allowed: this notebook is read-only."
    )
  return run_query(sql)


q(
    "SELECT DB_NAME() AS database_name, CURRENT_USER AS logged_in_as,"
    " SYSDATETIME() AS server_time"
)

,database_name,logged_in_as,server_time
0,assessmentDB,intern_readonly,2026-10-16 16:58:03.545668


In [3]:
try:
  q("DELETE FROM week06.london_2014")
except PermissionError as e:
  print("Blocked as expected ->", e)

Blocked as expected -> 'DELETE' is not allowed: this notebook is read-only.


In [4]:
conn = get_connection()
try:
  cur = conn.cursor()
  cur.execute("UPDATE week06.london_2014 SET [Mean TemperatureC] = 0 WHERE 1 = 0")
  print(
      "Unexpected: the server accepted the statement. Tell the admin the"
      " account isn't read-only."
  )
except Exception as e:
  print("Server refused, as expected ->", str(e)[:160])
finally:
  conn.rollback()
  conn.close()

Server refused, as expected -> (229, b"The UPDATE permission was denied on the object 'london_2014', database 'assessmentDB', schema 'week06'.DB-Lib error message 20018, severity 14:\nGeneral


In [5]:
tables = q("""
    SELECT TABLE_SCHEMA AS schema_name, TABLE_NAME AS table_name, TABLE_TYPE
    FROM INFORMATION_SCHEMA.TABLES
    ORDER BY TABLE_SCHEMA, TABLE_NAME
""")
print(len(tables), "tables")
tables

67 tables


,schema_name,table_name,TABLE_TYPE
0,sys,database_firewall_rules,VIEW
1,week06,beijing_pek_2014,BASE TABLE
2,week06,brasilia_bsb_2014,BASE TABLE
3,week06,capetown_cpt_2014,BASE TABLE
4,week06,delhi_del_2014,BASE TABLE
...,...,...,...
62,week19,shopping_center,BASE TABLE
63,week19,task1_kmeans_customer_data,BASE TABLE
64,week19,task2_hierarchical_plant_data,BASE TABLE
65,week19,task3_nlp_text_data,BASE TABLE


In [6]:
# Tables per schema
display(q("""
    SELECT TABLE_SCHEMA AS schema_name, COUNT(*) AS n_tables
    FROM INFORMATION_SCHEMA.TABLES
    GROUP BY TABLE_SCHEMA
    ORDER BY TABLE_SCHEMA
"""))

# Inspect columns of london_2014
display(q("""
    SELECT ORDINAL_POSITION AS pos, COLUMN_NAME, DATA_TYPE, IS_NULLABLE
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = 'week06' AND TABLE_NAME = 'london_2014'
    ORDER BY ORDINAL_POSITION
"""))

,schema_name,n_tables
0,sys,1
1,week06,8
2,week07,1
3,week08,3
4,week09,3
5,week14,10
6,week15,8
7,week16,9
8,week17,8
9,week18,7


,pos,COLUMN_NAME,DATA_TYPE,IS_NULLABLE
0,1,GMT,varchar,YES
1,2,Max TemperatureC,bigint,YES
2,3,Mean TemperatureC,bigint,YES
3,4,Min TemperatureC,bigint,YES
4,5,Dew PointC,bigint,YES
5,6,MeanDew PointC,bigint,YES
6,7,Min DewpointC,bigint,YES
7,8,Max Humidity,bigint,YES
8,9,Mean Humidity,bigint,YES
9,10,Min Humidity,bigint,YES


In [7]:
q("""
    SELECT DISTINCT '[' + COLUMN_NAME + ']' AS exact_name_to_use
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = 'week06' AND COLUMN_NAME LIKE ' %'      -- starts with a space
    ORDER BY exact_name_to_use
""")

,exact_name_to_use
0,[ CloudCover]
1,[ Events]
2,[ Max Gust SpeedKm/h]
3,[ Max Sea Level PressurehPa]
4,[ Max VisibilityKm]
5,[ Max Wind SpeedKm/h]
6,[ Mean Humidity]
7,[ Mean Sea Level PressurehPa]
8,[ Mean VisibilityKm]
9,[ Mean Wind SpeedKm/h]


In [8]:
q("""
    SELECT TABLE_SCHEMA + '.' + TABLE_NAME AS table_name, COLUMN_NAME, DATA_TYPE
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE COLUMN_NAME LIKE '%Income%' OR COLUMN_NAME = 'Age'
    ORDER BY table_name
""")

,table_name,COLUMN_NAME,DATA_TYPE
0,week14.assignment1_ecommerce_data,Age,float
1,week14.assignment1_ecommerce_data,Annual_Income,float
2,week14.data,Age,float
3,week14.housing_price_data,Age,bigint
4,week14.task1_data_with_missing,Age,float
5,week14.task1_data_with_missing,Income,float
6,week14.task2_categorical_data,Age,bigint
7,week14.task3_scaling_data,Age,bigint
8,week16.assignment1_house_prices,Age,bigint
9,week16.assignment2_marketing_campaign,Age,bigint


In [9]:
q("""
    SELECT
        COUNT(*)                                   AS n_rows,
        COUNT(DISTINCT Gender)                     AS n_gender_values,
        SUM(CASE WHEN Gender IS NULL THEN 1 ELSE 0 END)           AS missing_gender,
        SUM(CASE WHEN Applicant_Income IS NULL THEN 1 ELSE 0 END) AS missing_income,
        SUM(CASE WHEN Loan_Amount IS NULL THEN 1 ELSE 0 END)      AS missing_loan_amount,
        MIN(Applicant_Income)                      AS min_income,
        MAX(Applicant_Income)                      AS max_income
    FROM week17.loan_approval_prediction
""")

,n_rows,n_gender_values,missing_gender,missing_income,missing_loan_amount,min_income,max_income
0,500,2,0,0,0,17000,80000


In [10]:
q("SELECT TOP 5 * FROM week09.comtrade_milk_uk_monthly_14")

,Classification,Year,Period,Period Desc.,Aggregate Level,Is Leaf Code,Trade Flow Code,Trade Flow,Reporter Code,Reporter,Reporter ISO,Partner Code,Partner,Partner ISO,2nd Partner Code,2nd Partner,2nd Partner ISO,Customs Proc. Code,Customs,Mode of Transport Code,Mode of Transport,Commodity Code,Commodity,Qty Unit Code,Qty Unit,Qty,Alt Qty Unit Code,Alt Qty Unit,Alt Qty,Netweight (kg),Gross weight (kg),Trade Value (US$),CIF Trade Value (US$),FOB Trade Value (US$),Flag
0,HS,2014,201401,January 2014,4,0,1,Imports,826,United Kingdom,None,0,World,None,None,None,None,None,None,None,None,0401,"Milk and cream, neither concentrated nor sweet...",0,None,None,None,None,None,22404316,None,21950747,None,None,0
1,HS,2014,201401,January 2014,4,0,2,Exports,826,United Kingdom,None,0,World,None,None,None,None,None,None,None,None,0401,"Milk and cream, neither concentrated nor sweet...",0,None,None,None,None,None,60497363,None,46923551,None,None,0
2,HS,2014,201401,January 2014,4,0,2,Exports,826,United Kingdom,None,4,Afghanistan,None,None,None,None,None,None,None,None,0401,"Milk and cream, neither concentrated nor sweet...",0,None,None,None,None,None,2520,None,3410,None,None,0
3,HS,2014,201401,January 2014,4,0,2,Exports,826,United Kingdom,None,40,Austria,None,None,None,None,None,None,None,None,0401,"Milk and cream, neither concentrated nor sweet...",0,None,None,None,None,None,21,None,316,None,None,0
4,HS,2014,201401,January 2014,4,0,1,Imports,826,United Kingdom,None,56,Belgium,None,None,None,None,None,None,None,None,0401,"Milk and cream, neither concentrated nor sweet...",0,None,None,None,None,None,1978279,None,4472349,None,None,0


In [11]:
q("""
    SELECT 
        TABLE_SCHEMA AS schema_name,
        TABLE_NAME AS table_name,
        COUNT(*) AS n_columns
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA IN ('week16', 'week17')
    GROUP BY TABLE_SCHEMA, TABLE_NAME
    ORDER BY n_columns DESC
""")

,schema_name,table_name,n_columns
0,week16,customer_churn_prediction,19
1,week17,loan_approval_prediction,16
2,week16,assignment3_credit_risk,10
3,week17,assignment2_medical_diagnosis,10
4,week16,assignment2_marketing_campaign,9
5,week16,task3_classification_data,9
6,week16,assignment1_house_prices,8
7,week17,assignment1_product_recommendation,8
8,week17,assignment3_classifier_comparison,7
9,week17,task3_naive_bayes_data,7


In [12]:
q("""
    SELECT
        COUNT(*) AS n_rows,
        COUNT(DISTINCT Contract_Type) AS n_contract_types,
        SUM(CASE WHEN Churn = 1 THEN 1 ELSE 0 END) AS n_churned,
        MIN(Monthly_Charges) AS min_monthly_charges,
        MAX(Monthly_Charges) AS max_monthly_charges
    FROM week16.customer_churn_prediction
""")

,n_rows,n_contract_types,n_churned,min_monthly_charges,max_monthly_charges
0,500,3,228,20.03,119.20


In [13]:
q("""
    SELECT TOP 10
           country,
           ROUND([SP.POP.TOTL] / 1000.0, 1) AS pop_thousands
    FROM   week08.wb_pop_2013
    WHERE  [SP.POP.TOTL] IS NOT NULL
    ORDER BY [SP.POP.TOTL] ASC
""")

,country,pop_thousands
0,Tuvalu,9.90
1,Palau,20.90
2,St. Martin (French part),31.30
3,San Marino,31.40
4,Turks and Caicos Islands,33.10
5,Sint Maarten (Dutch part),36.60
6,Liechtenstein,37.00
7,Monaco,37.50
8,Faeroe Islands,48.30
9,Marshall Islands,52.80


In [14]:
q("""
    SELECT 
        Application_ID,
        Applicant_Income,
        Loan_Amount,
        Credit_History
    FROM week17.loan_approval_prediction
    WHERE Self_Employed = 'Yes'
      AND Previous_Loan_Status = 'Defaulted'
      AND Loan_Approved = 1
""")

,Application_ID,Applicant_Income,Loan_Amount,Credit_History


In [15]:
q("""
    SELECT COUNT(*) AS missing_count
    FROM week17.loan_approval_prediction
    WHERE Gender IS NULL
       OR Married IS NULL
       OR Self_Employed IS NULL
""")

,missing_count
0,0


In [16]:
q("""
    SELECT
        DATENAME(weekday, TRY_CONVERT(date, GMT)) AS weekday_name,
        COUNT(*)                                  AS wet_day_count
    FROM   week06.london_2014
    WHERE  Precipitationmm > 0
      AND  TRY_CONVERT(date, GMT) IS NOT NULL
    GROUP BY DATENAME(weekday, TRY_CONVERT(date, GMT))
    ORDER BY wet_day_count DESC
""")

,weekday_name,wet_day_count
0,Monday,30
1,Friday,29
2,Tuesday,28
3,Wednesday,24
4,Saturday,23
5,Thursday,19
6,Sunday,17


In [17]:
q("""
    SELECT 
        ROUND(AVG(1.0 * Churn) * 100, 1) AS churn_percentage
    FROM   week16.customer_churn_prediction
""")

,churn_percentage
0,45.60


In [18]:
q("""
    SELECT
        CASE
            WHEN Tenure < 12 THEN 'New (<12m)'
            WHEN Tenure < 36 THEN 'Established (12-36m)'
            ELSE 'Loyal (36m+)'
        END                                        AS tenure_group,
        COUNT(*)                                   AS customers,
        ROUND(100.0 * AVG(1.0 * Churn), 1)         AS churn_rate_pct
    FROM week16.customer_churn_prediction
    GROUP BY
        CASE
            WHEN Tenure < 12 THEN 'New (<12m)'
            WHEN Tenure < 36 THEN 'Established (12-36m)'
            ELSE 'Loyal (36m+)'
        END
    ORDER BY tenure_group
""")

,tenure_group,customers,churn_rate_pct
0,Established (12-36m),175,45.70
1,Loyal (36m+),252,42.10
2,New (<12m),73,57.50


In [19]:
q("""
    SELECT
        Property_Area,
        COUNT(*)                                                    AS applications,
        SUM(CASE WHEN Loan_Approved = 1 THEN 1 ELSE 0 END)          AS approved,
        SUM(CASE WHEN Loan_Approved = 0 THEN 1 ELSE 0 END)          AS rejected,
        SUM(CASE WHEN Loan_Approved = 1 AND Credit_History = 0 
                 THEN 1 ELSE 0 END)                                 AS approved_with_bad_credit
    FROM week17.loan_approval_prediction
    GROUP BY Property_Area
    ORDER BY Property_Area
""")

,Property_Area,applications,approved,rejected,approved_with_bad_credit
0,Rural,166,0,166,0
1,Semiurban,167,167,0,0
2,Urban,167,167,0,0


In [20]:
q("""
    SELECT
        Region,
        Payment_Method,
        COUNT(*)                       AS customer_count,
        ROUND(AVG(Avg_Order_Value), 2) AS avg_order_value
    FROM week19.customer_analytics_data
    GROUP BY
        Region,
        Payment_Method
    HAVING COUNT(*) > 25
    ORDER BY
        Region,
        Payment_Method
""")

,Region,Payment_Method,customer_count,avg_order_value
0,East,Credit Card,36,492.20
1,East,Debit Card,40,489.52
2,North,Credit Card,45,486.00
3,North,Debit Card,42,483.67
4,North,Digital Wallet,35,490.04
5,South,Credit Card,39,484.96
6,South,Digital Wallet,37,499.73
7,West,Credit Card,35,482.14
8,West,Debit Card,39,484.94
9,West,Digital Wallet,26,476.28


In [21]:
q("""
    SELECT
        CASE 
            WHEN GROUPING(Internet_Service) = 1 THEN 'ALL SERVICES' 
            ELSE Internet_Service 
        END                                                      AS internet_service,
        CASE 
            WHEN GROUPING(Contract_Type) = 1 THEN '(subtotal)' 
            ELSE Contract_Type 
        END                                                      AS contract_type,
        COUNT(*)                                                 AS churned_customers
    FROM week16.customer_churn_prediction
    WHERE Churn = 1
    GROUP BY ROLLUP(Internet_Service, Contract_Type)
    ORDER BY 
        GROUPING(Internet_Service), 
        Internet_Service, 
        GROUPING(Contract_Type), 
        Contract_Type
""")

,internet_service,contract_type,churned_customers
0,DSL,Month-to-Month,28
1,DSL,One Year,33
2,DSL,Two Year,13
3,DSL,(subtotal),74
4,Fiber Optic,Month-to-Month,44
5,Fiber Optic,One Year,32
6,Fiber Optic,Two Year,25
7,Fiber Optic,(subtotal),101
8,No,Month-to-Month,19
9,No,One Year,24


In [22]:
q("""
    SELECT TOP 10
        g.country,
        ROUND(g.[NY.GDP.MKTP.CD] / 1e9, 1)     AS gdp_billions,
        l.[SP.DYN.LE00.IN]                     AS life_expectancy
    FROM week08.wb_gdp_2013 AS g
    JOIN week08.wb_le_2013  AS l ON l.country = g.country
    WHERE g.[NY.GDP.MKTP.CD] < 50000000000
      AND g.[NY.GDP.MKTP.CD] IS NOT NULL
      AND l.[SP.DYN.LE00.IN] IS NOT NULL
    ORDER BY life_expectancy DESC
""")

,country,gdp_billions,life_expectancy
0,Iceland,15.40,83.12
1,Faeroe Islands,2.60,81.30
2,Malta,9.60,80.75
3,Bermuda,5.60,80.57
4,Slovenia,47.70,80.28
5,Lebanon,44.40,80.13
6,Cyprus,24.10,79.95
7,Costa Rica,49.20,79.23
8,Brunei Darussalam,16.10,78.55
9,Albania,12.80,77.54


In [23]:
q("""
    SELECT
        w.Country AS who_country_missing_le
    FROM week06.who_pop_tb_all AS w
    LEFT JOIN week08.wb_le_2013 AS l 
           ON l.country = w.Country
    WHERE l.[SP.DYN.LE00.IN] IS NULL
    ORDER BY w.Country
""")

,who_country_missing_le
0,Andorra
1,Bahamas
2,Bolivia (Plurinational State of)
3,Congo
4,Cook Islands
5,Côte d'Ivoire
6,Democratic People's Republic of Korea
7,Democratic Republic of the Congo
8,Dominica
9,Egypt


In [24]:
q("""
    SELECT
        city,
        MAX(max_temp) AS hottest_temp
    FROM (
        SELECT 'London'    AS city, TRY_CONVERT(date, GMT)    AS day, [Max TemperatureC] AS max_temp FROM week06.london_2014
        UNION ALL 
        SELECT 'Delhi',     TRY_CONVERT(date, [Date]),         [Max TemperatureC] FROM week06.delhi_del_2014
        UNION ALL 
        SELECT 'Moscow',    TRY_CONVERT(date, [Date]),         [Max TemperatureC] FROM week06.moscow_svo_2014
    ) AS combined_weather
    GROUP BY city
    ORDER BY hottest_temp DESC
""")

,city,hottest_temp
0,Delhi,38
1,Moscow,33
2,London,30


In [25]:
q("""
    SELECT
        Customer_ID,
        Contract_Type,
        Monthly_Charges,
        (SELECT ROUND(AVG(Monthly_Charges), 2)
           FROM week16.customer_churn_prediction AS i
          WHERE i.Contract_Type = o.Contract_Type) AS contract_avg_charge
    FROM week16.customer_churn_prediction AS o
    WHERE Churn = 1
      AND Monthly_Charges > (SELECT AVG(Monthly_Charges)
                               FROM week16.customer_churn_prediction AS i
                              WHERE i.Contract_Type = o.Contract_Type)
    ORDER BY Monthly_Charges DESC
""")

,Customer_ID,Contract_Type,Monthly_Charges,contract_avg_charge
0,CUST0217,Month-to-Month,119.20,64.11
1,CUST0486,Two Year,118.91,64.95
2,CUST0339,One Year,118.54,62.05
3,CUST0147,Month-to-Month,118.32,64.11
4,CUST0188,One Year,116.80,62.05
...,...,...,...,...
107,CUST0124,One Year,64.84,62.05
108,CUST0415,One Year,64.73,62.05
109,CUST0060,One Year,63.98,62.05
110,CUST0264,One Year,63.58,62.05


In [26]:
q("""
    WITH payment_stats AS (
        -- Step 1: Compute each payment method's customer count and churn rate
        SELECT
            Payment_Method,
            COUNT(*)                             AS total_customers,
            AVG(1.0 * Churn)                     AS churn_rate
        FROM week16.customer_churn_prediction
        GROUP BY Payment_Method
    ),
    overall_stats AS (
        -- Step 2: Compute the global overall churn rate
        SELECT
            AVG(1.0 * Churn)                     AS overall_churn_rate
        FROM week16.customer_churn_prediction
    )
    -- Step 3: Combine them, filter for methods above average, and compute the difference
    SELECT
        p.Payment_Method,
        ROUND(p.churn_rate * 100.0, 1)           AS method_churn_pct,
        ROUND(o.overall_churn_rate * 100.0, 1)   AS overall_churn_pct,
        ROUND((p.churn_rate - o.overall_churn_rate) * 100.0, 1) AS diff_percentage_points
    FROM payment_stats p
    CROSS JOIN overall_stats o
    WHERE p.churn_rate > o.overall_churn_rate
    ORDER BY diff_percentage_points DESC
""")

,Payment_Method,method_churn_pct,overall_churn_pct,diff_percentage_points
0,Electronic Check,49.30,45.60,3.70
1,Credit Card,48.10,45.60,2.50


In [27]:
q("""
    WITH ranked_customers AS (
        SELECT
            Preferred_Category                               AS category,
            Customer_ID                                      AS customer,
            Avg_Order_Value                                  AS value,
            ROW_NUMBER() OVER (PARTITION BY Preferred_Category 
                               ORDER BY Avg_Order_Value DESC) AS rank
        FROM week19.customer_analytics_data
    )
    SELECT
        category,
        customer,
        value,
        rank
    FROM ranked_customers
    WHERE rank <= 2
    ORDER BY
        category,
        rank
""")

,category,customer,value,rank
0,Beauty,CID_0074,500.00,1
1,Beauty,CID_0409,500.00,2
2,Books,CID_0489,500.00,1
3,Books,CID_0380,500.00,2
4,Electronics,CID_0498,500.00,1
5,Electronics,CID_0125,500.00,2
6,Fashion,CID_0092,500.00,1
7,Fashion,CID_0265,500.00,2
8,Home & Kitchen,CID_0407,500.00,1
9,Home & Kitchen,CID_0372,500.00,2


In [28]:
q("""
    WITH daily AS (
        SELECT 
            TRY_CONVERT(date, GMT) AS day,
            [Mean TemperatureC]    AS mean_temp
        FROM week06.london_2014
    ),
    with_lag AS (
        SELECT
            day,
            mean_temp,
            LAG(day)       OVER (ORDER BY day) AS prev_day,
            LAG(mean_temp) OVER (ORDER BY day) AS prev_temp,
            mean_temp - LAG(mean_temp) OVER (ORDER BY day) AS temp_change
        FROM daily
    )
    SELECT TOP 5
        prev_day   AS yesterday,
        day        AS today,
        prev_temp  AS yesterday_temp,
        mean_temp  AS today_temp,
        temp_change
    FROM with_lag
    WHERE prev_temp IS NOT NULL
    ORDER BY temp_change ASC
""")

,yesterday,today,yesterday_temp,today_temp,temp_change
0,2014-06-03,2014-06-04,16,11,-5
1,2014-11-02,2014-11-03,13,8,-5
2,2014-03-09,2014-03-10,12,8,-4
3,2014-03-16,2014-03-17,13,9,-4
4,2014-10-03,2014-10-04,17,13,-4


In [29]:
q("""
    WITH daily AS (
        SELECT 
            TRY_CONVERT(date, GMT) AS day,
            Precipitationmm        AS rain_mm
        FROM week06.london_2014
    ),
    running AS (
        SELECT
            day,
            SUM(rain_mm) OVER (
                PARTITION BY YEAR(day), MONTH(day)
                ORDER BY day
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS mtd_rain_mm
        FROM daily
    )
    SELECT
        day         AS month_end_date,
        mtd_rain_mm AS total_monthly_rainfall
    FROM running
    WHERE day = EOMONTH(day)
    ORDER BY day
""")

,month_end_date,total_monthly_rainfall
0,2014-01-31,88.38
1,2014-02-28,50.55
2,2014-03-31,15.74
3,2014-04-30,30.99
4,2014-05-31,44.69
5,2014-06-30,26.93
6,2014-07-31,34.04
7,2014-08-31,81.27
8,2014-09-30,2.54
9,2014-10-31,36.07


In [30]:
q("""
    WITH segment_stats AS (
        SELECT 
            Property_Area,
            Education,
            COUNT(*)                                   AS applications,
            ROUND(100.0 * AVG(1.0 * Loan_Approved), 1) AS approval_rate_pct
        FROM week17.loan_approval_prediction
        GROUP BY Property_Area, Education
    ),
    overall AS (
        SELECT ROUND(100.0 * AVG(1.0 * Loan_Approved), 1) AS overall_rate_pct
        FROM week17.loan_approval_prediction
    )
    SELECT 
        s.Property_Area,
        s.Education,
        s.applications,
        s.approval_rate_pct,
        ROUND(s.approval_rate_pct - o.overall_rate_pct, 1) AS gap_vs_overall_pts,
        RANK() OVER (ORDER BY s.approval_rate_pct DESC)    AS approval_rank
    FROM segment_stats s
    CROSS JOIN overall o
    ORDER BY approval_rank
""")

,Property_Area,Education,applications,approval_rate_pct,gap_vs_overall_pts,approval_rank
0,Semiurban,Graduate,167,100.00,33.20,1
1,Urban,Graduate,167,100.00,33.20,1
2,Rural,Not Graduate,166,0.00,-66.80,3


In [31]:
q("""
    SELECT TOP 10
        Application_ID,
        Property_Area,
        Loan_Amount,
        Credit_History,
        Loan_Approved,
        RANK() OVER (PARTITION BY Property_Area ORDER BY Loan_Amount DESC) AS rank_in_area
    FROM week17.loan_approval_prediction
    WHERE Credit_History = 0
      AND Loan_Approved = 1
      AND Loan_Amount IS NOT NULL
    ORDER BY Loan_Amount DESC
""")

,Application_ID,Property_Area,Loan_Amount,Credit_History,Loan_Approved,rank_in_area
